In [ ]:
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = 'YOUR_LANGSMITH_API_KEY'

In [ ]:
# Cell 1: Load the model and processor
from transformers import AutoTokenizer, AutoProcessor, LlavaForConditionalGeneration
import torch

# Model ID and device for DeepSeek-VL-2
model_name = "deepseek-ai/deepseek-vl-2-7b"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load processor and tokenizer for DeepSeek-VL-2
try:
    processor = AutoProcessor.from_pretrained(model_name, use_auth_token=os.environ.get('HF_TOKEN')) # Replace with your actual token if needed
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=os.environ.get('HF_TOKEN')) # Replace with your actual token if needed
    print("Processor and tokenizer loaded successfully.")
except Exception as e:
    print(f"Error loading processor or tokenizer: {e}")
    processor = None
    tokenizer = None

# Load DeepSeek-VL-2 model
if processor is not None and tokenizer is not None:
    try:
        model = LlavaForConditionalGeneration.from_pretrained(model_name, use_auth_token=os.environ.get('HF_TOKEN')).to(device) # Replace with your actual token if needed
        print(f"Model loaded successfully on {device}")
    except torch.cuda.OutOfMemoryError as e:
        print(f"Error loading model onto CUDA: {e}")
        print("Attempting to load model onto CPU. This will be slower.")
        device = torch.device("cpu")
        model = LlavaForConditionalGeneration.from_pretrained(model_name, use_auth_token=os.environ.get('HF_TOKEN')).to(device) # Replace with your actual token if needed
        print(f"Model loaded successfully on {device}")
    except Exception as e:
        print(f"An unexpected error occurred during model loading: {e}")
        model = None
else:
    model = None

In [ ]:
# Cell 2: Run the model on the toy example
from PIL import Image
import torch

# Ensure model, processor, and tokenizer are loaded from the previous cell
if 'model' not in locals() or model is None or 'processor' not in locals() or processor is None or 'tokenizer' not in locals() or tokenizer is None:
    print("Error: Model, processor, or tokenizer not loaded. Please run the first cell successfully.")
else:
    # Load image
    try:
        image = Image.open("./AlexWhite_3_25.png").convert("RGB")
    except FileNotFoundError:
        print("Error: The image file './AlexWhite_3_25.png' was not found.")
    except Exception as e:
        print(f"Error loading the image: {e}")
        exit()

    # Introduction prompt
    introduction_prompt = "You are an expert chemist and understand chemical mechanisms. You can answer questions about specific slides from chemistry lectures. Given this image, answer the following multiple-choice question. There is only one correct answer. Justify your answer."

    # Question
    question = """Synthesis of first generation catalysts is shown. What is the reaction that takes 36h?
a. Buchwald–Hartwig Coupling
b. Acryodone cyclization
c. Addition of mesityl nucleophile
d. Bromination"""

    # Combine the prompt and question
    query = f"{introduction_prompt} {question}"

    # Prepare inputs using the processor
    inputs = processor(images=image, text=query, return_tensors="pt").to(device)

    # Generate output
    try:
        output = model.generate(**inputs, max_new_tokens=512) # Increased max_new_tokens to allow for justification

        # Decode and print
        answer = tokenizer.decode(output[0], skip_special_tokens=True)
        print(answer)
    except torch.cuda.OutOfMemoryError as e:
        print(f"CUDA Out of Memory during generation: {e}")
        print("Consider reducing the batch size, max_new_tokens, or using a smaller model.")
    except Exception as e:
        print(f"An unexpected error occurred during generation: {e}")